# TROPOMI and MPAS with UXARRAY directly

First we need to import the driver.

In [ ]:
from melodies_monet import driver

In [ ]:
# Needed if you want to make changes to `melodies_monet` and don't want to restart kernel:
%load_ext autoreload

%autoreload 2

## Initiate the analysis class

Now lets create an instance of the {mod}`melodies_monet.driver` {class}`~melodies_monet.driver.analysis` class.
It consists of 4 main parts: model instances, observation instances, a paired instance of both.
This will allow us to move things around the plotting function for spatial and overlays and more complex plots.

In [ ]:
an = driver.analysis()
an

## Control File

Read in the required yaml control file that sets up all the definitions of what we want to pair and plot.

In [ ]:
an.control = '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/control_tropomi_l2_CO_mpas_20240101.yaml'
an.read_control()
an.control_dict

## Load the model data 

The driver will automatically loop through the "models" found in the model section of the control file and create model classes for each. Classes include the label, mapping information, and xarray object as well as the filenames.  Note it can open multiple files easily by including wildcards. Here we are only opening one CAM-chem file.

In [ ]:
an.open_models()

In [ ]:
an.models

In [ ]:
an.models['mpas'].obj

In [ ]:
# All the info in the model class can be called here.
print(an.models['mpas'].label)
print(an.models['mpas'].mapping)

In [ ]:
# All the info in the analysis class can also be called.
print(an.start_time)
print(an.end_time)
print(an.download_maps)

## Open Obs

Now for monet-analysis we will open preprocessed data in either netcdf icartt or some other format.  We will not be retrieving data like monetio does for some observations (ie aeronet, airnow, etc....).  Instead we will provide utitilies to do this so that users can add more data easily.

Like models we list all obs objects in the yaml file and it will loop through and create driver.observation instances that include the model type, file, objects (i.e. data object) and label  

In [ ]:
an.control_dict['obs']

In [ ]:
an.open_obs()

In [ ]:
# All the info in the observation class can also be called.
#an.obs['tempo_l2_no2'].obj
print(an.obs['tropomi_l2_no2'].regrid_method) 

## Pair model and obs data

In [ ]:
%%time
 
an.pair_data()

po = an.paired['tropomi_l2_no2_mpas'].obj
print("times:", po['time'].values)                       
for v in ['NO2', 'nitrogendioxide_tropospheric_column']:
    print(v, po[v].notnull().sum(dim='n_face').values)    

In [ ]:
an.paired

In [ ]:
an.paired['tropomi_l2_no2_mpas'].obj
# ds = an.paired['tropomi_l2_no2_mpas'].obj
# ds.to_netcdf("tropomi_l2_no2_mpas")

In [ ]:
df = an.paired['tropomi_l2_no2_mpas'].obj.to_dataframe()

In [ ]:
pair = an.paired['tropomi_l2_no2_mpas']  # or your pair key

## Generate plots

In [ ]:
%%time

an.plotting()

In [ ]:
gran = list(an.obs['tropomi_l2_no2'].obj.values())[0]
gran = gran[0] if isinstance(gran, list) else gran
print("time coord:", gran['time'].values)
print("vars:", list(gran.variables))
for v in gran.variables:
    if 'time' in v.lower() and v != 'time':
        a = gran[v].values
        print(v, "->", a.min(), a.max(), getattr(gran[v], 'attrs', {}))